In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
# Read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [9]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## Build Dataset

In [37]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
    
    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix] # crop and append
X = torch.tensor(X)
Y = torch.tensor(Y)

In [38]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

## Implementing the embedding lookup table

In [12]:
C = torch.randn((27, 2)) # character embedding matrix
emb = C[X] # embed the characters into vectors
emb.shape

torch.Size([32, 3, 2])

In [15]:
emb[0] # the first batch which was '...' is now three vectors of dimension 2, where each vector is the embedding of the character '.' (which has index 0 in the vocabulary)
# The same will be done for each batch, so we will get a 3D tensor of shape (number of batches, block_size, embedding_dim)

tensor([[-0.4043,  0.0770],
        [-0.4043,  0.0770],
        [-0.4043,  0.0770]])

## Implementing the hidden layer

In [16]:
w1 = torch.randn((6, 100)) # the weight matrix for the hidden layer
# We take 6 because we have 2 dimensions in the embedding, and a block size of 3, so we have 2*3=6 inputs to the hidden layer
b1 = torch.randn(100) # the bias for the hidden layer

In [ ]:
torch.cat(torch.unbind(emb, 1), dim=1).shape # less efficient

torch.Size([32, 6])

In [18]:
# Much better then above
emb.view(32, 6).shape

torch.Size([32, 6])

In [21]:
h = torch.tanh(emb.view(-1, 6) @ w1 + b1)

In [24]:
h

tensor([[ 0.7577, -0.9470,  0.6062,  ...,  0.9276,  0.9549, -0.6795],
        [ 0.8512, -0.9533,  0.3267,  ...,  0.9816,  0.9844, -0.7984],
        [ 0.9755, -0.3767,  0.9537,  ...,  0.9981,  0.0683,  0.8292],
        ...,
        [-0.9999,  0.7863, -0.6216,  ..., -0.6644,  0.8377, -0.7103],
        [ 0.2707, -0.9412, -0.9576,  ...,  0.9922,  0.9997, -0.4463],
        [-0.1805, -0.4530,  0.7225,  ...,  0.8573,  0.7653, -0.8790]])

## Implementing the output layer

In [25]:
w2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [26]:
logits = h @ w2 + b2
logits.shape

torch.Size([32, 27])

In [27]:
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
probs.shape

torch.Size([32, 27])

In [28]:
loss = -probs[torch.arange(len(Y)), Y].log().mean()
loss

tensor(13.8161)

### Summary

In [39]:
X.shape, Y.shape # dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [40]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g) # character embedding matrix
w1 = torch.randn((6, 100), generator=g) # the weight matrix for the character embedding to hidden layer
b1 = torch.randn(100, generator=g) # the bias for the hidden layer
w2 = torch.randn((100, 27), generator=g) # the weight matrix for the hidden layer to output layer
b2 = torch.randn(27, generator=g) # the bias for the output layer
parameters = [C, w1, b1, w2, b2]

In [41]:
sum(p.nelement() for p in parameters) # number of parameters in the model

3481

In [42]:
for p in parameters:
    p.requires_grad = True

In [52]:
for _ in range(100):
    
    # minibatch construct
    ix = torch.randint(0, len(X), (32,))
    
    # forward pass
    emb = C[X[ix]] # embed the characters into vectors
    h = torch.tanh(emb.view(-1, 6) @ w1 + b1) # (32, 100)
    logits = h @ w2 + b2 # (32, 27)
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdim=True)
    # loss = -probs[torch.arange(len(Y)), Y].log().mean()
    loss = F.cross_entropy(logits, Y[ix]) # this is more numerically stable than the above
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    lr = 0.1 # learning rate
    for p in parameters:
        p.data += -lr * p.grad
        
print(loss.item())

2.4197254180908203
